In [14]:
import jax
import jax.numpy as np
import pytreeclass as pytc
import zodiax as zdx
import optax

# Simple class
class Foo(pytc.TreeClass):
    x : float
    y : float
    
    def __init__(self, x, y):
        self.x = np.asarray(x, float)
        self.y = np.asarray(y, float)
    
    def model(self):
        return self.x * self.y

# Construct object
tree = Foo(1, 2)

# Define parameters to optimise
params = 'x'

# # Use zodiax to automate mapping of optimisers to leaves
# optim, opt_state = zdx.get_optimiser(tree, params, optax.adam(1))


false_mask = tree.at[...].set(False)
x_mask = false_mask.at["x"].set(True)

optim = optax.chain(optax.masked(optax.sgd(learning_rate=1), x_mask))
optim_state = optim.init(tree)

In [15]:
@jax.jit
@jax.value_and_grad
# @filter(params)
def loss_fn(pytree):
    return np.abs(pytree.model())**2

loss, grads = loss_fn(tree)

# apply the update
updates, opt_state = optim.update(grads, opt_state)
updated_tree = zdx.apply_updates(tree, updates)


In [17]:
tree

Foo(x=f32[](μ=1.00, σ=0.00, ∈[1.00,1.00]), y=f32[](μ=2.00, σ=0.00, ∈[2.00,2.00]))

In [16]:
updated_tree

Foo(x=f32[](μ=-7.00, σ=0.00, ∈[-7.00,-7.00]), y=f32[](μ=6.00, σ=0.00, ∈[6.00,6.00]))

In [ ]:
def filter():
    def wrapper(func):
        def inner_wrapper(pytree, *args, **kwargs):

            pytree = pytc.tree_unmask(pytree)

            # Unmask and call original function
            def unmask(masked):
                unmasked = pytc.tree_unmask(masked, bool_mask)
                return func(unmasked, *args, **kwargs)
            
            # Mask and call wrapped function
            masked_tree = pytc.tree_mask(pytree, bool_mask)
            return unmask(masked_tree)

        return inner_wrapper
    return wrapper

In [22]:
def filter(func):
    def wrapper(pytree, *args, **kwargs):
        result = func(pytc.tree_unmask(pytree), *args, **kwargs)
        return result
    return wrapper

def mask(parmas):
    def wrapper(func):
        def inner_wrapper(pytree, *args, **kwargs):
            # Unmask and call original function
            def unmask(masked):
                unmasked = pytc.tree_unmask(masked, bool_mask)
                return func(unmasked, *args, **kwargs)
            
            # Mask and call wrapped function
            masked_tree = pytc.tree_mask(pytree, bool_mask)
            return unmask(masked_tree)

        return inner_wrapper
    return wrapper

In [30]:
import jax
import jax.numpy as np
import pytreeclass as pytc
import zodiax as zdx
import optax

# Simple class
class Foo(pytc.TreeClass):
    x : float
    y : float
    
    def __init__(self, x, y):
        self.x = np.asarray(x, float)
        self.y = np.asarray(y, float)
    
    def model(self):
        return self.x * self.y

# Construct object
tree = Foo(1, 2)

# Define parameters to optimise
params = 'x'

mask = tree.at[...].set(True).at[params].set(False)
masked = pytc.tree_mask(tree, mask)
print(masked)

# optim = optax.chain(optax.masked(optax.sgd(learning_rate=1), x_mask))
optim = optax.sgd(learning_rate=1)
opt_state = optim.init(tree)

@jax.jit
@jax.value_and_grad
@filter
def loss_fn(pytree):
    # pytree = pytree.at['y'].set(4.)
    pytree = pytree.at['y'].apply(lambda x: 2*x)
    return np.abs(pytree.model())**2

loss, grads = loss_fn(masked)
# loss, grads = loss_fn(tree)
loss, grads

Foo(x=1.0, y=#2.0)


(Array(16., dtype=float32),
 Foo(x=f32[](μ=32.00, σ=0.00, ∈[32.00,32.00]), y=#f32[](μ=2.00, σ=0.00, ∈[2.00,2.00])))

In [61]:
import equinox as eqx

def set(pytree, parameters, values):
    if values is None:
        values = [None]
        if isinstance(parameters, str):
            parameters = [parameters]
    new_parameters, new_values = zdx.base._format(parameters, values)
    leaves_fn = lambda pytree: zdx.base._get_leaves(pytree, new_parameters)
    return eqx.tree_at(leaves_fn, pytree, new_values, is_leaf = lambda leaf: leaf is None)



            

def apply(transformations, parmas):
    def wrapper(func):
        
        def inner_wrapper(pytree, *args, **kwargs):

            # Build boolean mask
            false_mask = pytree.at[...].set(True)
            bool_mask = set(false_mask, params, False)
            masked_tree = pytc.tree_mask(pytree, bool_mask)
            print("Masked: ", masked_tree)

            # Unmask and call original function
            def tf_fn(masked):
                # unmasked = pytc.tree_unmask(masked, bool_mask)
                unmasked = pytc.tree_unmask(masked)
                return func(unmasked, *args, **kwargs)
            
            # Apply transformations
            for tf in transformations:
                tf_fn = tf(tf_fn)
            
            return tf_fn(masked_tree)

        return inner_wrapper
    return wrapper

In [65]:
import jax
import jax.numpy as np
import pytreeclass as pytc
import zodiax as zdx
import optax

# Simple class
class Foo(pytc.TreeClass):
    x : float
    y : float
    name : str
    
    def __init__(self, x, y):
        self.x = np.asarray(x, float)
        self.y = np.asarray(y, float)
        self.name = 'A Name'
    
    def model(self):
        return self.x * self.y
    
    def __name__(self):
        return "Test"

# Construct object
tree = Foo(1, 2)
print(tree)

# Define parameters to optimise
# params = 'x'
# params = 'y'
params = ['x', 'y']

# @apply((jax.jit, jax.value_and_grad), params)
@apply((jax.value_and_grad, jax.jit), params)
def loss_fn(pytree):
    pytree = pytree.at['y'].apply(lambda x: 2*x)
    return np.abs(pytree.model())**2

loss, grads = loss_fn(tree)
loss, grads

Foo(x=1.0, y=2.0, name=A Name)
Masked:  Foo(x=1.0, y=2.0, name=#A Name)


(Array(16., dtype=float32),
 Foo(
   x=f32[](μ=32.00, σ=0.00, ∈[32.00,32.00]), 
   y=f32[](μ=16.00, σ=0.00, ∈[16.00,16.00]), 
   name=#A Name
 ))

In [66]:
Foo.__name__

'Foo'

In [63]:
import jax
import jax.numpy as np
import pytreeclass as pytc
import zodiax as zdx
import optax

tree = Foo(1, 2)
print(tree)

# params = ['x', 'y']
params = 'x'

@apply((jax.jit,), params)
def update(pytree):
    pytree = pytree.at['y'].apply(lambda x: 2*x)
    return pytree

new_tree = update(tree)
print(new_tree)

Foo(x=1.0, y=2.0, name=A Name)
Masked:  Foo(x=1.0, y=#2.0, name=#A Name)


TypeError: Value 'A Name' with type <class 'str'> is not a valid JAX type

In [51]:
new_tree.name

AttributeError: 'tuple' object has no attribute 'name'

In [157]:
import pytreeclass as pytc
import equinox as eqx
import zodiax as zdx

class Foo(pytc.TreeClass):
# class Foo(eqx.Module):
# class Foo(zdx.Base):
    bar : None
    
    def __init__(self, bar):
        self.bar = bar
    
    def __call__(self):
        return self.bar()

class Bar(pytc.TreeClass):
# class Bar(eqx.Module):
# class Bar(zdx.Base):
    layers : None

    def __init__(self, layers):
        d = {}
        for i, layer in enumerate(layers):
            d[f'layer{i}'] = layer
        self.layers = d
    
    def __getattr__(self, key):
        if key in self.layers.keys():
            return self.layers[key]
        raise AttributeError(key)
    
    def __call__(self):
        y = 1
        for layer in self.layers.values():
            y = layer(y)
        return y

class Layer(pytc.TreeClass):
# class Layer(eqx.Module):
# class Layer(zdx.Base):
    x : None

    def __init__(self, x):
        self.x = np.asarray(x, float)
    
    def __call__(self, x):
        return self.x * x

layers = [
    Layer(1),
    Layer(2),
    Layer(3),
]

bar = Bar(layers)
foo = Foo(bar)

foo, foo()

(Foo(
   bar=Bar(
     layers={
       layer0:Layer(x=f32[](μ=1.00, σ=0.00, ∈[1.00,1.00])), 
       layer1:Layer(x=f32[](μ=2.00, σ=0.00, ∈[2.00,2.00])), 
       layer2:Layer(x=f32[](μ=3.00, σ=0.00, ∈[3.00,3.00]))
     }
   )
 ),
 Array(6., dtype=float32))

In [199]:
import jax
import jax.numpy as np
from functools import wraps 
import pytreeclass as pytc


def set(pytree, parameters, values):
    if values is None:
        values = [None]
        if isinstance(parameters, str):
            parameters = [parameters]
    new_parameters, new_values = zdx.base._format(parameters, values)
    leaves_fn = lambda pytree: zdx.base._get_leaves(pytree, new_parameters)
    return eqx.tree_at(leaves_fn, pytree, new_values, is_leaf = lambda leaf: leaf is None)


class Foo(pytc.TreeClass):
    x : None
    y : None
    
    def __init__(self, x, y):
        self.x = np.asarray(x, float)
        self.y = np.asarray(y, float)
    
    def __call__(self):
        return self.x * self.y


foo = Foo(1, 2)
foo

Foo(x=f32[](μ=1.00, σ=0.00, ∈[1.00,1.00]), y=f32[](μ=2.00, σ=0.00, ∈[2.00,2.00]))

In [206]:
def filter(params):
    def decorator(func):
        def wrapper(pytree, *args, **kwargs):
            # Generate Mask
            false_mask = pytree.at[...].set(True)
            bool_mask = set(false_mask, params, False)
            masked_tree = pytc.tree_mask(pytree, bool_mask)

            def unmask_and_call(masked_tree, *args, **kwargs):
                unmasked_tree = pytc.tree_unmask(masked_tree)
                return func(unmasked_tree, *args, **kwargs)
            return unmask_and_call(masked_tree, *args, **kwargs)
        return wrapper
    return decorator

params = 'x'

# @jax.jit
@jax.value_and_grad
@filter(params)
def loss_fn(pytree):
    return np.abs(pytree() - 8)**2

loss, grads = loss_fn(foo)

print(loss)
print(grads)

36.0
Foo(x=-24.0, y=-12.0)


In [202]:
def filter(pytree, params):

    # Generate Mask
    false_mask = pytree.at[...].set(True)
    bool_mask = set(false_mask, params, False)
    masked_tree = pytc.tree_mask(pytree, bool_mask)


    def wrapper(func):
        def inner_wrapper(pytree, *args, **kwargs):
            unmasked = pytc.tree_unmask(pytree)
            return func(unmasked, *args, **kwargs)

        return inner_wrapper(masked_tree)
    return wrapper

params = 'x'

# @jax.jit
# @jax.value_and_grad
@filter(foo, params)
def loss_fn(pytree):
    return np.abs(pytree() - 8)**2

loss, grads = loss_fn(foo)

print(loss)
print(grads)

TypeError: 'ArrayImpl' object is not callable

In [176]:
def filter(params):
    def wrapper(func):
        
        @wraps(func)
        def inner_wrapper(pytree, *args, **kwargs):

            # Generate Mask
            false_mask = pytree.at[...].set(True)
            bool_mask = set(false_mask, params, False)

            # Unmask and call original function
            def unmask(masked):
                unmasked = pytc.tree_unmask(masked, bool_mask)
                return func(unmasked, *args, **kwargs)
            
            # Mask and call wrapped function
            masked_tree = pytc.tree_mask(pytree, bool_mask)
            return unmask(masked_tree)

        return inner_wrapper
    return wrapper

params = 'x'

@jax.jit
@jax.value_and_grad
@filter(params)
def loss_fn(pytree):
    return np.abs(pytree() - 8)**2

loss, grads = loss_fn(foo)

print(loss)
print(grads)

36.0
Foo(x=-24.0, y=-12.0)


In [183]:
def filter(params):
    def decorator(func):
        def wrapper(pytree, *args, **kwargs):

            # Generate Mask
            false_mask = pytree.at[...].set(True)
            bool_mask = set(false_mask, params, False)
            masked_tree = pytc.tree_mask(pytree, bool_mask)

            # Generate unmasked func
            def unmask(masked):
                unmasked = pytc.tree_unmask(masked, bool_mask)
                return func(unmasked, *args, **kwargs)

            # Mask and call wrapped function
            return unmask(masked_tree)

        return wrapper
    return decorator

params = 'x'

@jax.jit
@jax.value_and_grad
@filter(params)
def loss_fn(pytree):
    return np.abs(pytree())**2

loss, grads = loss_fn(foo)

print(loss)
print(grads)

4.0
Foo(x=8.0, y=4.0)


In [184]:
def filter(params):
    def decorator(func):
        def wrapper(pytree, *args, **kwargs):

            # Generate Mask
            false_mask = pytree.at[...].set(True)
            bool_mask = set(false_mask, params, False)
            masked_tree = pytc.tree_mask(pytree, bool_mask)

            # Generate unmasked func
            def unmask(masked):
                unmasked = pytc.tree_unmask(masked)
                return func(unmasked, *args, **kwargs)

            # Mask and call wrapped function
            return unmask(masked_tree)

        return wrapper
    return decorator

params = 'x'

@jax.jit
@jax.value_and_grad
@filter(params)
def loss_fn(pytree):
    return np.abs(pytree())**2

loss, grads = loss_fn(foo)

print(loss)
print(grads)

4.0
Foo(x=8.0, y=4.0)


In [191]:
from functools import partial

@partial(jax.jit, static_argnums = 1)
def update(pytree, param, value):
    unmasked = pytc.tree_unmask(pytree, mask)
    pytree_out = unmasked.at[param].set(value)
    return pytree_out

foo = Foo(1, 2)
mask = foo.at[...].set(True).at['x'].set(False)
masked_pyree = pytc.tree_mask(foo, mask)
update(masked_pyree, 'x', 3).x

Array(3, dtype=int32, weak_type=True)

In [192]:
import pytreeclass as pytc
from typing import Any
import jax
import jax.numpy as jnp


class ArrayValidator(pytc.TreeClass):
    def __init__(self, shape, dtype):
        """Validate shape and dtype of input array.

        Args:
            shape: Expected shape of array. available values are int, None, ...
                use int for fixed size, None for any size, and ... for any number
                of dimensions. for example (..., 1) allows any number of dimensions
                with the last dimension being 1. (1, ..., 1) allows any number of
                dimensions with the first and last dimensions being 1.
            dtype: Expected dtype of array.

        Example:
            >>> x = jnp.ones((5, 5))
            >>> # any number of dimensions with last dim=5
            >>> shape = (..., 5)
            >>> dtype = jnp.float32
            >>> validator = ArrayValidator(shape, dtype)
            >>> validator(x)  # no error

            >>> # must be 2 dimensions with first dim unconstrained and last dim=5
            >>> shape = (None, 5)
            >>> validator = ArrayValidator(shape, dtype)
            >>> validator(x)  # no error
        """

        if shape.count(...) > 1:
            raise ValueError("Only one ellipsis allowed")

        for si in shape:
            if not isinstance(si, (int, type(...), type(None))):
                raise TypeError(f"Expected int or ..., got {si}")

        self.shape = shape
        self.dtype = dtype

    def __call__(self, x):
        if not (hasattr(x, "shape") and hasattr(x, "dtype")):
            raise TypeError(f"Expected array with shape {self.shape}, got {x}")

        shape = list(self.shape)
        array_shape = list(x.shape)
        array_dtype = x.dtype

        if self.shape and array_dtype != self.dtype:
            raise TypeError(f"Dtype mismatch, {array_dtype=} != {self.dtype=}")

        if ... in shape:
            index = shape.index(...)
            shape = (
                shape[:index]
                + [None] * (len(array_shape) - len(shape) + 1)
                + shape[index + 1 :]
            )

        if len(shape) != len(array_shape):
            raise ValueError(f"{len(shape)=} != {len(array_shape)=}")

        for i, (li, ri) in enumerate(zip(shape, array_shape)):
            if li is None:
                continue
            if li != ri:
                raise ValueError(f"Size mismatch, {li} != {ri} at dimension {i}")
        return x


# any number of dimensions with firt dim=3 and last dim=6
shape = (3, ..., 6)
# dtype must be float32
dtype = jnp.float32

validator = ArrayValidator(shape=shape, dtype=dtype)

# convert to half precision from float32
converter = lambda x: x.astype(jnp.float16)


@pytc.autoinit
class Tree(pytc.TreeClass):
    array: jax.Array = pytc.field(callbacks=[validator, converter])


x = jnp.ones([3, 1, 2, 6])
tree = Tree(array=x)


try:
    y = jnp.ones([1, 1, 2, 3])
    tree = Tree(array=y)
except ValueError as e:
    print(e, "\n")
    # On applying ArrayValidator(shape=(3, Ellipsis, 6), dtype=<class 'jax.numpy.float32'>) for field=`array`:
    # Dtype mismatch, array_dtype=dtype('float16') != self.dtype=<class 'jax.numpy.float32'>

try:
    z = x.astype(jnp.float16)
    tree = Tree(array=z)
except TypeError as e:
    print(e)
    # On applying ArrayValidator(shape=(3, Ellipsis, 6), dtype=<class 'jax.numpy.float32'>) for field=`array`:
    # Size mismatch, 3 != 1 at dimension 0

On applying ArrayValidator(shape=(3, Ellipsis, 6), dtype=<class 'jax.numpy.float32'>) for field=`array`:
Size mismatch, 3 != 1 at dimension 0 

On applying ArrayValidator(shape=(3, Ellipsis, 6), dtype=<class 'jax.numpy.float32'>) for field=`array`:
Dtype mismatch, array_dtype=dtype('float16') != self.dtype=<class 'jax.numpy.float32'>


In [197]:
# x = jnp.ones([3, 1, 2, 6])
x = jnp.ones([3, 1, 2, 6])
tree = Tree(array=x)

tree.at['array'].set(jnp.ones([3, 1, 2, 7]).astype(jnp.float16))

Tree(array=f16[3,1,2,7](μ=1.00, σ=0.00, ∈[1.00,1.00]))

In [122]:
import jax
import jax.numpy as np
from functools import wraps 

# def my_decorator(params):
#     def decorator(func):
#         # Convert parameters
#         false_mask = pytree.at[...].set(False)
#         bool_mask = set(false_mask, params, True)
#         def wrapper(pytree, *args, **kwargs):
#             unmasked = pytc.tree_unmask(pytree, bool_mask)
#             return func(unmasked, *args, **kwargs)
#         return wrapper(pytc.tree_mask(pytree, bool_mask))
#     return decorator

def my_decorator(params):
    def decorator(func):
        # Convert parameters
        false_mask = pytree.at[...].set(False)
        bool_mask = set(false_mask, params, True)
        def mask_unmask(pytree):
            return pytc.tree_mask(pytree, bool_mask)
        def wrapper(pytree, *args, **kwargs):
            unmasked = pytc.tree_unmask(pytree, bool_mask)
            return func(unmasked, *args, **kwargs)
        return wrapper(pytc.tree_mask(pytree, bool_mask))
    return decorator


params = ['bar.layer1.x', 'bar.layer2.x']

@jax.grad
@my_decorator(params)
def loss(pytree):
    return np.abs(foo() - 8)**2

grads = loss(foo)

print(grads)

NameError: name 'pytree' is not defined

In [116]:
foo()

6.0

In [114]:
formatted = format(params, None)
print(get_where(grads, formatted[0])(grads))


[Array(0., dtype=float32, weak_type=True), Array(0., dtype=float32, weak_type=True)]


In [ ]:


def filter_grad(
    parameters : Params, 
    *filter_args,
    **filter_kwargs
    ) -> Callable:
    """
    Applies the equinox filter_grad function to the input parameters. The 
    corresponding equinox docs are found [here](https://docs.kidger.site/
    equinox/api/filtering/transformations/)

    Parameters
    ----------
    parameters : Union[str, List[str]]
        The parameters to filter. Can either be a single string path or a list
        of paths.
    *filter_args : Any
        The args to pass to the equinox filter_grad function.
    **filter_kwargs : Any
        The kwargs to pass to the equinox filter_grad function.
    
    Returns
    -------
    Callable
        The wrapped function.
    """
    def wrapper(func : Callable):
        
        @wraps(func)
        def inner_wrapper(pytree : PyTree, *args, **kwargs):

            # Convert parameters
            boolean_filter = zodiax.tree.boolean_filter(pytree, parameters)

            # Wrap original function
            @equinox.filter_grad(*filter_args, **filter_kwargs)
            def recombine(traced : PyTree, static : PyTree):
                return func(eqx.combine(traced, static), *args, **kwargs)
            
            # Return wrapped function
            return recombine(*eqx.partition(pytree, boolean_filter))
        return inner_wrapper
    return wrapper